# Week 38, Day 5: Project: tests and refactoring

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w038d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('errors.py', 'w', encoding='utf-8') as f:
    f.write('"""Every error the library raises on purpose."""\n\n\nclass LibraryError(Exception):\n    """Base class for the library\'s own errors."""\n\n\nclass InvalidInput(LibraryError, ValueError):\n    """Something typed or loaded doesn\'t make sense."""\n\n\nclass NotFound(LibraryError, LookupError):\n    """There\'s no book or member with that id."""\n\n\nclass NotAllowed(LibraryError):\n    """The library\'s rules don\'t allow this right now."""\n\n\nclass StorageError(LibraryError):\n    """The library\'s data couldn\'t be read or saved."""\n')
with open('values.py', 'w', encoding='utf-8') as f:
    f.write('"""Reading and checking the values people type, and showing money."""\nfrom errors import InvalidInput\n\nKINDS = {"B": "book", "M": "member"}\n\n\ndef clean_isbn(text):\n    """\'978-0-14-118776-1\' as \'9780141187761\'. Raises InvalidInput."""\n    shown = text.strip()\n    digits = shown.replace("-", "").replace(" ", "")\n    if len(digits) != 13 or not (digits.isascii() and digits.isdigit()):\n        raise InvalidInput(f"an ISBN has 13 digits: {shown!r}")\n    total = sum(int(d) * (3 if i % 2 else 1) for i, d in enumerate(digits))\n    if total % 10 != 0:\n        raise InvalidInput(f"wrong check digit: {shown!r}")\n    return digits\n\n\ndef clean_text(text, what, longest):\n    """\'  Things   Fall Apart \' as \'Things Fall Apart\'. Raises InvalidInput."""\n    cleaned = " ".join(text.split())\n    if not cleaned:\n        raise InvalidInput(f"{what} is missing")\n    if len(cleaned) > longest:\n        raise InvalidInput(f"{what} is longer than {longest} characters")\n    return cleaned\n\n\ndef parse_id(text, kind):\n    """\' b12 \' as \'B12\', for kind \'B\'. Raises InvalidInput."""\n    cleaned = text.strip().upper()\n    number = cleaned[1:]\n    if not cleaned.startswith(kind) or not (number.isascii() and number.isdigit()) or number.startswith("0"):\n        raise InvalidInput(f"not a {KINDS[kind]} id: {text.strip()!r}")\n    return cleaned\n\n\ndef parse_pence(text):\n    """\'£1.20\' or \'0.6\' as whole pence. Raises InvalidInput."""\n    shown = text.strip()\n    pounds, dot, pence = shown.removeprefix("£").partition(".")\n    if (not (pounds.isascii() and pounds.isdigit())\n            or (dot and not (pence.isascii() and pence.isdigit() and len(pence) <= 2))):\n        raise InvalidInput(f"not an amount: {shown!r}")\n    return int(pounds) * 100 + (int(pence.ljust(2, "0")) if dot else 0)\n\n\ndef format_pence(pence):\n    """60 as \'£0.60\', -350 as \'-£3.50\'."""\n    sign = "-" if pence < 0 else ""\n    pounds, rest = divmod(abs(pence), 100)\n    return f"{sign}£{pounds:,}.{rest:02d}"\n\n\ndef check_all(**checks):\n    """Run each check (a function of no arguments); return their results by name, or raise one\n    InvalidInput listing every problem, in order."""\n    results, problems = {}, []\n    for name, check in checks.items():\n        try:\n            results[name] = check()\n        except InvalidInput as err:\n            problems.append(str(err))\n    if problems:\n        raise InvalidInput("; ".join(problems))\n    return results\n')
with open('domain.py', 'w', encoding='utf-8') as f:
    f.write('"""The library\'s things and rules. No input, output or storage here."""\nfrom dataclasses import dataclass, field\nfrom datetime import date, timedelta\n\nfrom errors import InvalidInput, NotAllowed\nfrom values import format_pence\n\nLOAN_DAYS = 21\nFINE_PER_DAY = 20      # pence for each day late\nMAX_FINE = 500         # the most one loan can cost\nFINE_LIMIT = 500       # owing this much stops a member borrowing\n\n\n@dataclass\nclass Book:\n    book_id: str\n    isbn: str\n    title: str\n    author: str\n    borrower: str | None = None\n    holds: list[str] = field(default_factory=list)\n\n    @property\n    def on_loan(self):\n        return self.borrower is not None\n\n    def describe(self):\n        return f"{self.book_id}: {self.title} by {self.author}"\n\n\n@dataclass\nclass Member:\n    member_id: str\n    name: str\n    max_loans: int = 3\n    owes: int = 0\n\n\n@dataclass\nclass Loan:\n    loan_id: int\n    book_id: str\n    member_id: str\n    out: date\n    due: date\n    returned: date | None = None\n\n    @property\n    def is_open(self):\n        return self.returned is None\n\n    def days_late(self, today):\n        end = self.returned or today\n        return max(0, (end - self.due).days)\n\n    def fine(self, today):\n        return min(self.days_late(today) * FINE_PER_DAY, MAX_FINE)\n\n\ndef check_can_borrow(member, book, open_loans, today):\n    """Raise NotAllowed with the first reason the member can\'t borrow the book today."""\n    if book.on_loan:\n        raise NotAllowed(f"{book.book_id} is on loan")\n    if book.holds and book.holds[0] != member.member_id:\n        raise NotAllowed(f"{book.book_id} is being kept for someone else")\n    if len(open_loans) >= member.max_loans:\n        raise NotAllowed(f"{member.name} already has {len(open_loans)} books")\n    if any(loan.days_late(today) for loan in open_loans):\n        raise NotAllowed(f"{member.name} has an overdue book")\n    if member.owes >= FINE_LIMIT:\n        raise NotAllowed(f"{member.name} owes {format_pence(member.owes)}")\n\n\ndef lend(book, member, open_loans, today, loan_id):\n    """Lend the book, or raise NotAllowed. Returns the new Loan."""\n    check_can_borrow(member, book, open_loans, today)\n    if book.holds:\n        book.holds.pop(0)\n    book.borrower = member.member_id\n    return Loan(loan_id, book.book_id, member.member_id, today, today + timedelta(days=LOAN_DAYS))\n\n\ndef give_back(book, loan, member, today):\n    """Close the loan and add any fine to what the member owes. Returns the fine."""\n    assert loan.is_open and loan.book_id == book.book_id and loan.member_id == member.member_id\n    loan.returned = today\n    book.borrower = None\n    fine = loan.fine(today)\n    member.owes += fine\n    return fine\n\n\ndef reserve(book, member):\n    """Join the queue for a book. Returns the member\'s place in it."""\n    if not book.on_loan and not book.holds:\n        raise NotAllowed(f"{book.book_id} is on the shelf: borrow it instead")\n    if book.borrower == member.member_id:\n        raise NotAllowed(f"{member.name} already has {book.book_id}")\n    if member.member_id in book.holds:\n        raise NotAllowed(f"{member.name} is already waiting for {book.book_id}")\n    book.holds.append(member.member_id)\n    return len(book.holds)\n\n\ndef pay(member, pence):\n    """Take a payment towards the member\'s fines. Returns what they still owe."""\n    if pence <= 0:\n        raise InvalidInput("a payment must be more than £0")\n    if pence > member.owes:\n        raise NotAllowed(f"{member.name} only owes {format_pence(member.owes)}")\n    member.owes -= pence\n    return member.owes\n')
with open('store.py', 'w', encoding='utf-8') as f:
    f.write('"""Where the library\'s data is kept: in memory, or in a JSON file."""\nimport json\nimport logging\nimport os\nfrom dataclasses import asdict\nfrom datetime import date\n\nfrom domain import Book, Loan, Member\nfrom errors import NotFound, StorageError\n\nlog = logging.getLogger("shelf.store")\n\n\nclass MemoryStore:\n    """Keeps everything in dictionaries. commit() does nothing: there\'s nowhere to save."""\n\n    def __init__(self):\n        self.books = {}\n        self.members = {}\n        self.loans = []\n        self.counters = {"B": 0, "M": 0, "L": 0}\n\n    def next_id(self, kind):\n        """\'B1\', \'B2\', ... for books, \'M1\', ... for members, and 1, 2, ... for loans. Never reused."""\n        self.counters[kind] += 1\n        number = self.counters[kind]\n        return number if kind == "L" else f"{kind}{number}"\n\n    def add_book(self, book):\n        self.books[book.book_id] = book\n\n    def add_member(self, member):\n        self.members[member.member_id] = member\n\n    def add_loan(self, loan):\n        self.loans.append(loan)\n\n    def book(self, book_id):\n        try:\n            return self.books[book_id]\n        except KeyError:\n            raise NotFound(f"no book {book_id}") from None\n\n    def member(self, member_id):\n        try:\n            return self.members[member_id]\n        except KeyError:\n            raise NotFound(f"no member {member_id}") from None\n\n    def open_loans(self, member_id=None):\n        return [loan for loan in self.loans\n                if loan.is_open and (member_id is None or loan.member_id == member_id)]\n\n    def open_loan_for(self, book_id):\n        for loan in self.loans:\n            if loan.is_open and loan.book_id == book_id:\n                return loan\n        raise NotFound(f"{book_id} isn\'t on loan")\n\n    def commit(self):\n        pass\n\n\ndef to_data(store):\n    """Everything in the store as plain dictionaries and lists, ready for JSON."""\n    def loan_data(loan):\n        d = asdict(loan)\n        d["out"], d["due"] = loan.out.isoformat(), loan.due.isoformat()\n        d["returned"] = loan.returned.isoformat() if loan.returned else None\n        return d\n    return {\n        "books": [asdict(b) for b in store.books.values()],\n        "members": [asdict(m) for m in store.members.values()],\n        "loans": [loan_data(loan) for loan in store.loans],\n        "counters": dict(store.counters),\n    }\n\n\ndef fill(store, data):\n    """Load data made by to_data into an empty store."""\n    for d in data["books"]:\n        store.add_book(Book(**d))\n    for d in data["members"]:\n        store.add_member(Member(**d))\n    for d in data["loans"]:\n        d = dict(d)\n        d["out"], d["due"] = date.fromisoformat(d["out"]), date.fromisoformat(d["due"])\n        d["returned"] = date.fromisoformat(d["returned"]) if d["returned"] else None\n        store.add_loan(Loan(**d))\n    store.counters = dict(data["counters"])\n\n\nclass JsonStore(MemoryStore):\n    """A MemoryStore that loads from a JSON file when it\'s made, and saves on commit()."""\n\n    def __init__(self, path):\n        super().__init__()\n        self.path = path\n        self._load()\n\n    def _load(self):\n        try:\n            with open(self.path, encoding="utf-8") as f:\n                data = json.load(f)\n        except FileNotFoundError:\n            log.info("no file at %s yet: starting empty", self.path)\n            return\n        except (OSError, ValueError) as err:\n            raise StorageError(f"can\'t read {self.path}") from err\n        try:\n            fill(self, data)\n        except (KeyError, TypeError, ValueError) as err:\n            raise StorageError(f"{self.path} is damaged") from err\n        log.info("loaded %d books, %d members and %d loans from %s",\n                 len(self.books), len(self.members), len(self.loans), self.path)\n\n    def commit(self):\n        """Save everything safely: write a temporary file, then replace the old file in one step."""\n        temp = self.path + ".tmp"\n        try:\n            with open(temp, "w", encoding="utf-8") as f:\n                json.dump(to_data(self), f, indent=1, ensure_ascii=False)\n            os.replace(temp, self.path)\n        except OSError as err:\n            raise StorageError(f"can\'t save to {self.path}") from err\n        finally:\n            if os.path.exists(temp):\n                os.remove(temp)\n        log.debug("saved %s", self.path)\n')
with open('service.py', 'w', encoding='utf-8') as f:
    f.write('"""The library\'s use cases. Takes and returns values; never prints."""\nimport logging\n\nfrom domain import Book, Member, check_can_borrow, give_back, lend, pay, reserve\nfrom values import check_all, clean_isbn, clean_text, parse_id, parse_pence\n\nlog = logging.getLogger("shelf.service")\n\n\nclass Library:\n    def __init__(self, store, today):\n        self.store = store\n        self.today = today          # a function returning today\'s date\n\n    def book(self, book_id):\n        return self.store.book(parse_id(book_id, "B"))\n\n    def member(self, member_id):\n        return self.store.member(parse_id(member_id, "M"))\n\n    def add_book(self, isbn, title, author):\n        v = check_all(isbn=lambda: clean_isbn(isbn),\n                      title=lambda: clean_text(title, "title", 100),\n                      author=lambda: clean_text(author, "author", 60))\n        book = Book(self.store.next_id("B"), v["isbn"], v["title"], v["author"])\n        self.store.add_book(book)\n        self.store.commit()\n        log.info("added %s", book.book_id)\n        return book\n\n    def join(self, name):\n        member = Member(self.store.next_id("M"), clean_text(name, "name", 60))\n        self.store.add_member(member)\n        self.store.commit()\n        log.info("added %s", member.member_id)\n        return member\n\n    def lend(self, book_id, member_id):\n        book, member = self.book(book_id), self.member(member_id)\n        open_loans, today = self.store.open_loans(member.member_id), self.today()\n        check_can_borrow(member, book, open_loans, today)    # first, so a refusal uses up no loan id\n        loan = lend(book, member, open_loans, today, self.store.next_id("L"))\n        self.store.add_loan(loan)\n        self.store.commit()\n        log.info("lent %s to %s", book.book_id, member.member_id)\n        return loan\n\n    def give_back(self, book_id):\n        book = self.book(book_id)\n        loan = self.store.open_loan_for(book.book_id)\n        fine = give_back(book, loan, self.store.member(loan.member_id), self.today())\n        self.store.commit()\n        log.info("%s returned, fine %d", book.book_id, fine)\n        return loan, fine\n\n    def reserve(self, book_id, member_id):\n        place = reserve(self.book(book_id), self.member(member_id))\n        self.store.commit()\n        return place\n\n    def pay(self, member_id, amount):\n        left = pay(self.member(member_id), parse_pence(amount))\n        self.store.commit()\n        return left\n\n    def overdue(self):\n        """(loan, days late) for every late loan, most days late first, then by book id."""\n        today = self.today()\n        late = [(loan, loan.days_late(today)) for loan in self.store.open_loans() if loan.days_late(today)]\n        return sorted(late, key=lambda pair: (-pair[1], pair[0].book_id))\n\n    def find(self, words):\n        """Books whose title or author contains every word (in any case), by id number."""\n        wanted = words.lower().split()\n        found = [b for b in self.store.books.values()\n                 if all(w in f"{b.title} {b.author}".lower() for w in wanted)]\n        return sorted(found, key=lambda b: int(b.book_id[1:]))\n\n    def loans(self, member_id):\n        """The member, and their open loans in the order they were made."""\n        member = self.member(member_id)\n        return member, self.store.open_loans(member.member_id)\n')
with open('cli.py', 'w', encoding='utf-8') as f:
    f.write('"""The text interface: turns typed lines into calls, and results into text."""\nimport shlex\n\nfrom errors import LibraryError\nfrom values import format_pence\n\nUSAGE = {\n    "add": ("ISBN TITLE AUTHOR", 3),\n    "join": ("NAME", 1),\n    "lend": ("BOOK MEMBER", 2),\n    "return": ("BOOK", 1),\n    "reserve": ("BOOK MEMBER", 2),\n    "pay": ("MEMBER AMOUNT", 2),\n    "overdue": ("", 0),\n    "find": ("WORDS", None),\n}\n\n\ndef handle(line, library):\n    """One typed line in, one piece of text out."""\n    try:\n        words = shlex.split(line)\n    except ValueError as err:\n        return f"error: {err}"\n    if not words:\n        return ""\n    command, args = words[0].lower(), words[1:]\n    if command not in USAGE:\n        return f"unknown command: {command}"\n    shape, count = USAGE[command]\n    if (count is None and not args) or (count is not None and len(args) != count):\n        return f"usage: {command} {shape}".rstrip()\n    try:\n        return execute(command, args, library)\n    except LibraryError as err:\n        return f"error: {err}"\n\n\ndef execute(command, args, library):\n    """Carry out one command, and describe the result."""\n    if command == "add":\n        return f"added {library.add_book(*args).describe()}"\n    if command == "join":\n        member = library.join(args[0])\n        return f"joined {member.member_id}: {member.name}"\n    if command == "lend":\n        loan = library.lend(*args)\n        return f"{loan.book_id} lent to {library.member(loan.member_id).name}, due {loan.due.isoformat()}"\n    if command == "return":\n        loan, fine = library.give_back(args[0])\n        text = f"{loan.book_id} returned"\n        if fine:\n            text += f" {loan.days_late(loan.returned)} days late: fine {format_pence(fine)}"\n        holds = library.book(loan.book_id).holds\n        if holds:\n            text += f"; keep it for {library.member(holds[0]).name}"\n        return text\n    if command == "reserve":\n        place = library.reserve(*args)\n        book, member = library.book(args[0]), library.member(args[1])\n        return f"{member.name} is number {place} in the queue for {book.book_id}"\n    if command == "pay":\n        left = library.pay(*args)\n        return f"{library.member(args[0]).name} now owes {format_pence(left)}"\n    if command == "overdue":\n        late = library.overdue()\n        if not late:\n            return "nothing is overdue"\n        return "\\n".join(f"{loan.book_id} {library.book(loan.book_id).title}: "\n                         f"{library.member(loan.member_id).name}, {days} days late" for loan, days in late)\n    if command == "find":\n        books = library.find(" ".join(args))\n        if not books:\n            return "no books found"\n        return "\\n".join(b.describe() + (" (on loan)" if b.on_loan else "") for b in books)\n')
with open('app.py', 'w', encoding='utf-8') as f:
    f.write('"""The composition root: builds the program from settings, and runs it."""\nimport logging\nimport sys\nfrom datetime import date\n\nfrom cli import handle\nfrom service import Library\nfrom store import JsonStore, MemoryStore\n\n\ndef build(env, today=date.today):\n    """A Library wired up from settings (a dictionary such as os.environ)."""\n    level = env.get("SHELF_LOG", "WARNING").upper()\n    logging.basicConfig(level=getattr(logging, level, logging.WARNING),\n                        format="%(levelname)s %(name)s: %(message)s", stream=sys.stderr)\n    path = env.get("SHELF_DATA", "library.json")\n    store = MemoryStore() if path == ":memory:" else JsonStore(path)\n    return Library(store, today)\n\n\ndef main(lines, env, today=date.today):\n    library = build(env, today)\n    for line in lines:\n        reply = handle(line, library)\n        if reply:\n            print(reply)\n')

### Exercise: Step 25: test the domain

Write a pytest suite for the rules in `domain.py`. Use at least one fixture and `pytest.raises` (with `match=` to check it's the right refusal), and a parametrized test if it suits. It must pass on the real `domain.py` and catch all **eight** broken versions.

In [ ]:
from datetime import date

import pytest

from domain import (FINE_LIMIT, MAX_FINE, Book, Loan, Member, check_can_borrow,
                    give_back, lend, reserve)
from errors import NotAllowed

TODAY = date(2026, 11, 2)


@pytest.fixture
def book():
    return Book("B1", "9780141187761", "Things Fall Apart", "Chinua Achebe")


___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"domain.py\").read()\nBROKEN = {\n    'due day counts as late': GOOD.replace('        return max(0, (end - self.due).days)', '        return max(0, (end - self.due).days + 1)'),\n    'no cap on fines': GOOD.replace('        return min(self.days_late(today) * FINE_PER_DAY, MAX_FINE)', '        return self.days_late(today) * FINE_PER_DAY'),\n    'queue ignored': GOOD.replace('    if book.holds and book.holds[0] != member.member_id:\\n        raise NotAllowed(f\"{book.book_id} is being kept for someone else\")\\n', ''),\n    'one book too many': GOOD.replace('    if len(open_loans) >= member.max_loans:', '    if len(open_loans) > member.max_loans:'),\n    'owing exactly the limit allowed': GOOD.replace('    if member.owes >= FINE_LIMIT:', '    if member.owes > FINE_LIMIT:'),\n    'refusal changes the queue': GOOD.replace('    check_can_borrow(member, book, open_loans, today)\\n    if book.holds:\\n        book.holds.pop(0)\\n', '    if book.holds:\\n        book.holds.pop(0)\\n    check_can_borrow(member, book, open_loans, today)\\n'),\n    'returned book still out': GOOD.replace('    loan.returned = today\\n    book.borrower = None\\n', '    loan.returned = today\\n'),\n    'reserving twice': GOOD.replace('    if member.member_id in book.holds:\\n        raise NotAllowed(f\"{member.name} is already waiting for {book.book_id}\")\\n', ''),\n}\ndef test_style():\n    \"a fixture, parametrize or pytest.raises\"\n    assert \"@pytest.fixture\" in source and \"pytest.raises\" in source\ndef test_catches():\n    \"your tests pass on the real code and catch all 8 broken versions\"\n    for name, text in BROKEN.items():\n        assert text != GOOD, name\n    results = against(source, \"domain\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"domain.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the real code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Step 26: test the service

Write a pytest suite for `service.py`, using a `MemoryStore` and a clock you control (a function returning a date you can change). To check that the service commits, use a small subclass of `MemoryStore` that counts calls to `commit`, as Week 34 did with fakes. It must pass on the real `service.py` and catch all **six** broken versions.

In [ ]:
from datetime import date

import pytest

from errors import NotAllowed
from service import Library
from store import MemoryStore


class CountingStore(MemoryStore):
    ___


@pytest.fixture
def lib():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"service.py\").read()\nBROKEN = {\n    \"lend doesn't commit\": GOOD.replace('        self.store.add_loan(loan)\\n        self.store.commit()\\n', '        self.store.add_loan(loan)\\n'),\n    'refusal uses up a loan id': GOOD.replace('        check_can_borrow(member, book, open_loans, today)    # first, so a refusal uses up no loan id\\n', ''),\n    'overdue in book order': GOOD.replace('key=lambda pair: (-pair[1], pair[0].book_id)', 'key=lambda pair: pair[0].book_id'),\n    'find matches any word': GOOD.replace('if all(w in f', 'if any(w in f'),\n    'find sorts ids as text': GOOD.replace('key=lambda b: int(b.book_id[1:])', 'key=lambda b: b.book_id'),\n    'ids must be typed exactly': GOOD.replace('        return self.store.book(parse_id(book_id, \"B\"))', '        return self.store.book(book_id)'),\n}\ndef test_style():\n    \"a fixture, parametrize or pytest.raises\"\n    assert \"@pytest.fixture\" in source and \"pytest.raises\" in source\ndef test_catches():\n    \"your tests pass on the real code and catch all 6 broken versions\"\n    for name, text in BROKEN.items():\n        assert text != GOOD, name\n    results = against(source, \"service\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"service.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the real code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Step 27: a table of commands

Refactor `cli.py` (it's in the editor) so that each command is a small function, `do_add(library, isbn, title, author)`, `do_join(library, name)`, ..., `do_find(library, *words)`, each returning the reply; and one dictionary, `COMMANDS`, maps each command to `(shape, count, function)`, replacing both `USAGE` and the `if command == ...` chain. `handle` looks the command up and calls `function(library, *args)`. Every reply must stay **exactly** the same: the checks replay a long session through your `handle` and through the original, and compare.

In [ ]:
"""The text interface: turns typed lines into calls, and results into text."""
import shlex

from errors import LibraryError
from values import format_pence

USAGE = {
    "add": ("ISBN TITLE AUTHOR", 3),
    "join": ("NAME", 1),
    "lend": ("BOOK MEMBER", 2),
    "return": ("BOOK", 1),
    "reserve": ("BOOK MEMBER", 2),
    "pay": ("MEMBER AMOUNT", 2),
    "overdue": ("", 0),
    "find": ("WORDS", None),
}


def handle(line, library):
    """One typed line in, one piece of text out."""
    try:
        words = shlex.split(line)
    except ValueError as err:
        return f"error: {err}"
    if not words:
        return ""
    command, args = words[0].lower(), words[1:]
    if command not in USAGE:
        return f"unknown command: {command}"
    shape, count = USAGE[command]
    if (count is None and not args) or (count is not None and len(args) != count):
        return f"usage: {command} {shape}".rstrip()
    try:
        return execute(command, args, library)
    except LibraryError as err:
        return f"error: {err}"


def execute(command, args, library):
    """Carry out one command, and describe the result."""
    if command == "add":
        return f"added {library.add_book(*args).describe()}"
    if command == "join":
        member = library.join(args[0])
        return f"joined {member.member_id}: {member.name}"
    if command == "lend":
        loan = library.lend(*args)
        return f"{loan.book_id} lent to {library.member(loan.member_id).name}, due {loan.due.isoformat()}"
    if command == "return":
        loan, fine = library.give_back(args[0])
        text = f"{loan.book_id} returned"
        if fine:
            text += f" {loan.days_late(loan.returned)} days late: fine {format_pence(fine)}"
        holds = library.book(loan.book_id).holds
        if holds:
            text += f"; keep it for {library.member(holds[0]).name}"
        return text
    if command == "reserve":
        place = library.reserve(*args)
        book, member = library.book(args[0]), library.member(args[1])
        return f"{member.name} is number {place} in the queue for {book.book_id}"
    if command == "pay":
        left = library.pay(*args)
        return f"{library.member(args[0]).name} now owes {format_pence(left)}"
    if command == "overdue":
        late = library.overdue()
        if not late:
            return "nothing is overdue"
        return "\n".join(f"{loan.book_id} {library.book(loan.book_id).title}: "
                         f"{library.member(loan.member_id).name}, {days} days late" for loan, days in late)
    if command == "find":
        books = library.find(" ".join(args))
        if not books:
            return "no books found"
        return "\n".join(b.describe() + (" (on loan)" if b.on_loan else "") for b in books)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nimport cli as _before\nfrom service import Library as _Lib\nfrom store import MemoryStore as _Mem\nSESSION = ['add 978-0-14-118776-1 \"Things Fall Apart\" \"Chinua Achebe\"', 'add 9780435950828 \"Arrow of God\" \"Chinua Achebe\"', 'add 1 \"\" \"\"', 'join \"Ada Obi\"', 'join \"Bo Lee\"', 'join', 'lend b1 m1', 'lend B1 M2', 'reserve B1 M2', 'reserve B2 M1', 'find achebe', 'find', 'find \"of god\"', 'overdue', 'overdue x', 'frob', 'lend B9 M1', 'pay M1 1', '', 'join \"Ada', '@later', 'overdue', 'return B1', 'lend B1 M1', 'lend B1 M2', 'pay m1 \u00a30.20', 'pay M1 x', 'return B2', 'RETURN b1', 'find dickens']\ndef replies(handler):\n    day = [_dt.date(2026, 11, 2)]\n    lib = _Lib(_Mem(), lambda: day[0])\n    out = []\n    for line in SESSION:\n        if line == \"@later\":\n            day[0] = _dt.date(2026, 11, 26)\n            continue\n        out.append(handler(line, lib))\n    return out\n\ndef test_same():\n    \"every reply is the same as before\"\n    got, want = replies(handle), replies(_before.handle)\n    for line, g, w in zip([l for l in SESSION if l != \"@later\"], got, want):\n        assert g == w, f\"{line!r}: got {g!r}, expected {w!r}\"\ndef test_table():\n    \"a table, not a chain\"\n    assert \"if command ==\" not in source, \"no if-chain on the command\"\n    assert \"USAGE\" not in source and \"def execute\" not in source, \"COMMANDS replaces USAGE and execute\"\n    assert len(COMMANDS) == 8 and all(len(v) == 3 and callable(v[2]) for v in COMMANDS.values())\n    assert COMMANDS[\"lend\"][:2] == (\"BOOK MEMBER\", 2) and COMMANDS[\"find\"][:2] == (\"WORDS\", None)")

### Exercise: Step 28: one more command

The service now has a query `loans(member_id)`, returning the member and their open loans (it's in `service.py`). Add a command `loans MEMBER` to the refactored interface: one line per open loan, `B1 Things Fall Apart, due 2026-11-23`, in the order they were lent, or `Ada Obi has no books`. Notice how little you have to change.

In [ ]:
"""The text interface: turns typed lines into calls, and results into text."""
import shlex

from errors import LibraryError
from values import format_pence


def do_add(library, isbn, title, author):
    return f"added {library.add_book(isbn, title, author).describe()}"


def do_join(library, name):
    member = library.join(name)
    return f"joined {member.member_id}: {member.name}"


def do_lend(library, book_id, member_id):
    loan = library.lend(book_id, member_id)
    return f"{loan.book_id} lent to {library.member(loan.member_id).name}, due {loan.due.isoformat()}"


def do_return(library, book_id):
    loan, fine = library.give_back(book_id)
    text = f"{loan.book_id} returned"
    if fine:
        text += f" {loan.days_late(loan.returned)} days late: fine {format_pence(fine)}"
    holds = library.book(loan.book_id).holds
    if holds:
        text += f"; keep it for {library.member(holds[0]).name}"
    return text


def do_reserve(library, book_id, member_id):
    place = library.reserve(book_id, member_id)
    book, member = library.book(book_id), library.member(member_id)
    return f"{member.name} is number {place} in the queue for {book.book_id}"


def do_pay(library, member_id, amount):
    left = library.pay(member_id, amount)
    return f"{library.member(member_id).name} now owes {format_pence(left)}"


def do_overdue(library):
    late = library.overdue()
    if not late:
        return "nothing is overdue"
    return "\n".join(f"{loan.book_id} {library.book(loan.book_id).title}: "
                     f"{library.member(loan.member_id).name}, {days} days late" for loan, days in late)


def do_find(library, *words):
    books = library.find(" ".join(words))
    if not books:
        return "no books found"
    return "\n".join(b.describe() + (" (on loan)" if b.on_loan else "") for b in books)


COMMANDS = {
    "add": ("ISBN TITLE AUTHOR", 3, do_add),
    "join": ("NAME", 1, do_join),
    "lend": ("BOOK MEMBER", 2, do_lend),
    "return": ("BOOK", 1, do_return),
    "reserve": ("BOOK MEMBER", 2, do_reserve),
    "pay": ("MEMBER AMOUNT", 2, do_pay),
    "overdue": ("", 0, do_overdue),
    "find": ("WORDS", None, do_find),
}


def handle(line, library):
    """One typed line in, one piece of text out."""
    try:
        words = shlex.split(line)
    except ValueError as err:
        return f"error: {err}"
    if not words:
        return ""
    command, args = words[0].lower(), words[1:]
    if command not in COMMANDS:
        return f"unknown command: {command}"
    shape, count, action = COMMANDS[command]
    if (count is None and not args) or (count is not None and len(args) != count):
        return f"usage: {command} {shape}".rstrip()
    try:
        return action(library, *args)
    except LibraryError as err:
        return f"error: {err}"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import datetime as _dt\nfrom service import Library as _Lib\nfrom store import MemoryStore as _Mem\ndef test_loans():\n    \"the loans command\"\n    day = [_dt.date(2026, 11, 2)]\n    lib = _Lib(_Mem(), lambda: day[0])\n    for line in ['add 978-0-14-118776-1 \"Things Fall Apart\" \"Chinua Achebe\"', 'add 9780435950828 \"Arrow of God\" \"Chinua Achebe\"', 'join \"Ada Obi\"']:\n        handle(line, lib)\n    assert handle(\"loans M1\", lib) == \"Ada Obi has no books\"\n    handle(\"lend B2 M1\", lib)\n    day[0] = _dt.date(2026, 11, 5)\n    handle(\"lend B1 M1\", lib)\n    assert handle(\" loans m1 \", lib) == \"B2 Arrow of God, due 2026-11-23\\nB1 Things Fall Apart, due 2026-11-26\"\n    handle(\"return B2\", lib)\n    assert handle(\"loans M1\", lib) == \"B1 Things Fall Apart, due 2026-11-26\"\n    assert handle(\"loans\", lib) == \"usage: loans MEMBER\" and handle(\"loans M7\", lib) == \"error: no member M7\"\n    assert handle(\"find arrow\", lib) == \"B2: Arrow of God by Chinua Achebe\"")

> **Tests that can fail.** Every test suite this week was graded against broken versions. That's the real test of a test: a suite that passes on everything proves nothing. When you write tests for your own programs, try breaking the code on purpose (change a `>=` to `>`, delete a line) and check a test fails. If none does, you've found a gap. Tools called mutation testers do exactly this automatically, making hundreds of small changes and reporting the ones no test noticed.

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

Every module so far is in the files below. One small addition: `service.py` has a new query, `loans(member_id)`, for Step 28.

## What to test, and where

Each layer gets the kind of test that suits it:

| Layer | Test it with | Why |
|---|---|---|
| domain | direct calls on objects made in the test | the rules are pure: objects in, objects out, no clock, no files |
| service | a `MemoryStore` (or a counting subclass) and a clock you control | fast, no files, and you can choose the day |
| store | `tmp_path` (Week 24) and real files | the files are the point |
| interface | whole lines through `handle` | that's what a volunteer sees |

Most tests should be at the bottom, where they're quickest and most precise, and fewer at the top, where one test covers a lot but tells you less about where a failure is. This shape is often called the test pyramid (from Mike Cohn's *Succeeding with Agile*).

The most valuable tests sit at the **edges** of rules, where off-by-one mistakes live: a book returned **on** its due date, a member with exactly their limit, owing exactly £5. Each broken version below is a mistake of that kind.

## Step 25: test the domain

## Step 26: test the service

The service's job is to put things together: read ids, ask the domain, commit. So its tests check exactly that: that typed ids work, that it commits, that it sorts and filters as promised. They don't re-test every rule: that's what Step 25 did.

## Step 27: a table of commands

`execute` in `cli.py` is a chain of eight `if`s, and `USAGE` repeats the list of commands in a second place. Adding a command means editing both, and the chain grows with every feature. Week 35 met the cure: a **dictionary of functions** (the Command pattern's simplest form). One table holds each command's shape, its argument count and the function that carries it out.

Refactoring means changing the structure **without** changing the behaviour. Before refactoring code that has no tests of its own, pin its current behaviour down with a **characterisation test**: run it on a varied set of inputs, record what it does, and check the new version does the same. (The name comes from Michael Feathers' *Working Effectively with Legacy Code*.) That's how this step is checked: a long session goes through your `handle` and through the original, and every reply must match.

## Step 28: one more command

Two edits: one function, one entry in the table. The `if` chain would have needed a new branch and a new `USAGE` line, and a careless edit could have broken a neighbouring command. That's the payoff of today's refactoring, and the tests are what made it safe to do.

## Quiz

## Before you go

Put the two test files in `tests/` in your `shelf` folder, run `pytest`, and commit. Then do the refactoring and the new command on your own copy, running the tests after each small change, and commit again. Tomorrow: the project review.